In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import time
import calendar

In [2]:
# Set up WebDriver
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run in background (remove if debugging)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [3]:
# Open the insurance page
driver.get("https://www.income.com.sg/buy/travel-insurance")

# Wait for form elements to load
wait = WebDriverWait(driver, 10)

In [4]:
step_num = 0

def screenshot(phase, y=0):
    time.sleep(1)
    driver.execute_script(f"window.scrollBy(0, {y});")

    global step_num
    driver.save_screenshot(f"#{step_num} {phase}.png")

    step_num += 1
    print(f"========== {phase} DONE ==========")

def click_button(by_what, name):
    element = wait.until(EC.presence_of_element_located((by_what, name)))
    driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", element)
    button = wait.until(EC.element_to_be_clickable((by_what, name)))
    driver.execute_script("arguments[0].click();", button)

In [5]:
# Select policy type
def select_policy_type(policy_type):
    if (policy_type == "per-trip"):
        policy_id = "evgTripPolicy"
    elif (policy_type == "yearly"):
        policy_id = "evgYearlyPolicy"
    else:
        raise Exception("Invalid policy type: " + policy_type)
    
    click_button(By.ID, policy_id)

    # TODO: haven't done yearly

select_policy_type("per-trip")

screenshot("policy", 600)

========== policy DONE ==========


In [6]:
# Select coverage type
def select_coverage_type(coverage_type):
    if (coverage_type == "individual"):
        coverage_id = "Individual/Group"
    elif (coverage_type == "family"):
        coverage_id = "Family"
    else:
        raise Exception("Invalid coverage type: " + coverage_type)
    
    click_button(By.XPATH, f"//li[@data-target='coverageType' and @data-value='{coverage_id}']")

    # TODO: haven't done family

select_coverage_type("individual")

screenshot("coverage", 200)

========== coverage DONE ==========


In [7]:
# Select destination
def select_destination(destination):
    click_button(By.XPATH, "//input[contains(@placeholder, 'Select destination(s)')]")

    if (destination == 'one-or-more'):
        region = 'One or more countries'
    elif (destination == 'asean'):
        region = 'ASEAN'
    elif (destination == 'asia'):
        region = 'Asia'
    elif (destination == 'worldwide'):
        region = 'Worldwide'
    else:
        raise Exception("Invalid destination: " + destination)

    click_button(By.CSS_SELECTOR, f"label[data-value='{region}']")

    if (destination == 'asean' or destination == 'asia' or destination == 'worldwide'):
        click_button(By.XPATH, "//button[contains(text(), 'I UNDERSTAND')]")

    # TODO: one-or-more not done

select_destination('asia')

screenshot('destination_2')

TimeoutException: Message: 


In [ ]:
def date_splitter(date):
    day, month, year = date.split('/')
    month_name = calendar.month_name[int(month)]  # e.g., 5 -> "May"
    return day, month_name, year

def click_calendar(date):
    target_date, target_month, target_year = date_splitter(date)

    wait = WebDriverWait(driver, 10)
    calendar_display = wait.until(EC.visibility_of_element_located((By.ID, 'ui-datepicker-div')))

    # find correct month & year
    while True:
        displayed_month = calendar_display.find_element(By.CLASS_NAME, "ui-datepicker-month").text
        displayed_year = calendar_display.find_element(By.CLASS_NAME, "ui-datepicker-year-no-dropdown").text

        if displayed_month == target_month and displayed_year == target_year:
            break

        click_button(By.CLASS_NAME, "ui-datepicker-next")
        time.sleep(0.5)

    # find correct date
    click_button(By.XPATH, f".//a[@data-date='{target_date}']")

In [ ]:
# Input departure date
def select_departure_date(departure_date):
    click_button(By.CSS_SELECTOR, '[data-test="c_startDate"]')
    click_calendar(departure_date)

select_departure_date('22/05/2025')

screenshot("departure_date")

TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x0123D363+60275]
	GetHandleVerifier [0x0123D3A4+60340]
	(No symbol) [0x010706F3]
	(No symbol) [0x010B8690]
	(No symbol) [0x010B8A2B]
	(No symbol) [0x01100EE2]
	(No symbol) [0x010DD0D4]
	(No symbol) [0x010FE6EB]
	(No symbol) [0x010DCE86]
	(No symbol) [0x010AC623]
	(No symbol) [0x010AD474]
	GetHandleVerifier [0x01488FE3+2467827]
	GetHandleVerifier [0x014845E6+2448886]
	GetHandleVerifier [0x0149F80C+2560028]
	GetHandleVerifier [0x01253DF5+153093]
	GetHandleVerifier [0x0125A3BD+179149]
	GetHandleVerifier [0x01244BB8+91080]
	GetHandleVerifier [0x01244D60+91504]
	GetHandleVerifier [0x0122FA10+4640]
	BaseThreadInitThunk [0x763B7BA9+25]
	RtlInitializeExceptionChain [0x7792C2EB+107]
	RtlClearBits [0x7792C26F+191]


In [ ]:
# Input arriving date
def select_arriving_date(arriving_date):
    click_button(By.CSS_SELECTOR, '[data-test="c_endDate"]')
    click_calendar(arriving_date)

select_arriving_date('29/05/2025')

screenshot("arriving_date")

========== arriving_date DONE ==========


In [ ]:
# # Select left Singapore option
# left_singapore = True

# if left_singapore:
#     yes_btn = driver.find_element(By.ID, "hasDeparted")
#     yes_btn.click()
# else:
#     no_btn = driver.find_element(By.ID, "notDeparted")
#     no_btn.click()

# screenshot("left_singapore")

In [ ]:
# Click consent checkbox
def click_consent_checkbox():
    click_button(By.XPATH, "//span[@class='checkmark']")

click_consent_checkbox()

screenshot("consent_box", 1000)

========== consent_box DONE ==========


In [ ]:
# Click "Get Quote" button
def click_get_quote():
    click_button(By.CSS_SELECTOR, '[data-test="b_quote"]')
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "per-trip-policy-plans")))

click_get_quote()

screenshot("quote")

========== quote DONE ==========


In [ ]:
def get_plan_data():
    plan_data = []

    # Locate all plan columns
    plan_columns = driver.find_elements(By.CLASS_NAME, "plan-item-header")

    for plan in plan_columns:
        plan_info = {}

        # Get plan name
        plan_name_element = plan.find_element(By.CLASS_NAME, "plan-name")
        plan_name_text = plan_name_element.text.strip().split("\n")[0]

        if not plan_name_text:
            continue

        plan_info["plan_name"] = plan_name_text

        # Get prices
        price_sections = plan.find_elements(By.CLASS_NAME, "th-2col")

        if len(price_sections) >= 2:
            # Adult price
            adult_price = price_sections[0].find_element(By.CLASS_NAME, "th-item-left").find_elements(By.TAG_NAME, "div")[1].text.strip()
            plan_info["price_per_adult"] = adult_price

            # Child price
            child_price = price_sections[1].find_element(By.CLASS_NAME, "th-item-left").find_elements(By.TAG_NAME, "div")[1].text.strip()
            plan_info["price_per_child"] = child_price
        else:
            # In case child price is not available
            plan_info["price_per_adult"] = "N/A"
            plan_info["price_per_child"] = "N/A"

        plan_data.append(plan_info)

    return plan_data

print(get_plan_data())

[{'plan_name': 'Classic', 'price_per_adult': '$90.19', 'price_per_child': '$68.90'}, {'plan_name': 'Deluxe', 'price_per_adult': '$111.36', 'price_per_child': '$82.66'}, {'plan_name': 'Preferred', 'price_per_adult': '$152.54', 'price_per_child': '$109.48'}]
